In [ ]:
# =============================================================
# 01_download_sources.ipynb
# Step 1 
# ------------------------------------------------------------
# What this does
# ──────────────
# 1. Loads configs from configs/indicators.yml + configs/pipeline.yml
# 2. Fetches ALL World Bank non-aggregate countries + income groups
# 3. Pulls data from three sources for all countries, 2015–2024:
#      WDI  — World Bank World Development Indicators (default API)
#      WGI  — Worldwide Governance Indicators (source=75)
#      UIS  — UNESCO Institute for Statistics
#               API mode  : requires UIS_API_KEY env variable
#               BDDS mode : bulk CSV download, no key needed
#              Configure via pipeline.yml  →  uis.mode: "api" | "bdds"
# 4. Merges all sources; WDI values fill gaps where UIS is primary
#    (indicators with wb_fallback in indicators.yml)
# 5. Builds a long-format year-by-year availability matrix
# 6. Computes a per-country coverage summary and wide matrix
# 7. Ranks countries within each income tier by overall coverage
# 8. Saves all outputs to outputs/coverage_analytics/
#
# Usage
# ─────
#   # With UIS API key:
#   export UIS_API_KEY="your-subscription-key"
#   python src/01_download_sources.py
#
#   # Without UIS key (BDDS bulk download):
#   # Set uis.mode: "bdds" in configs/pipeline.yml, then:
#   python src/01_download_sources.py
#
# Outputs
# ───────
#   outputs/coverage_analytics/
#     raw_long_2015_2024.csv              all rows from all sources
#     coverage_long_2015_2024.csv         year-by-year avail (0/1)
#     coverage_by_indicator_2015_2024.csv per-country-indicator summary
#     coverage_matrix_2015_2024.csv       wide matrix (proportions)
#     coverage_matrix_pct_2015_2024.csv   wide matrix (percentages)
#     top3_by_income_group_2015_2024.csv  top-3 per income tier
#     top10_by_income_group_2015_2024.csv top-10 per income tier
#     coverage_matrix_2015_2024.xlsx      multi-sheet workbook
#     data_source_log_2015_2024.csv       rows-per-indicator-source log
#
# Requirements
# ────────────
#   pip install requests pandas numpy openpyxl pyyaml
# =============================================================

from __future__ import annotations

import sys
from pathlib import Path

# Notebook-safe path setup: works in .py files AND Jupyter cells
try:
    _SRC = Path(__file__).resolve().parent
except NameError:
    # __file__ is undefined in Jupyter — walk up from cwd to find src/
    _cwd = Path.cwd()
    _SRC = _cwd / 'src' if (_cwd / 'src').exists() else _cwd
if str(_SRC) not in sys.path:
    sys.path.insert(0, str(_SRC))

import numpy as np
import pandas as pd

from utils import (
    find_project_root,
    load_indicators,
    load_pipeline,
    by_source,
    clip_map,
    sort_by_income,
    wb_get_countries,
    build_iso2_map,
    wb_fetch_indicator,
    uis_fetch,
    deduplicate,
    INCOME_ORDER,
)


# ──────────────────────────────────────────────────────────────
# Setup
# ──────────────────────────────────────────────────────────────
PROJECT_ROOT = find_project_root()
IND_CFG      = load_indicators()
PIPE_CFG     = load_pipeline()

YEAR_MIN   = PIPE_CFG["years"]["min"]
YEAR_MAX   = PIPE_CFG["years"]["max"]
YEARS      = list(range(YEAR_MIN, YEAR_MAX + 1))
N_YEARS    = len(YEARS)
PERIOD_TAG = f"{YEAR_MIN}_{YEAR_MAX}"

WB_CFG  = PIPE_CFG["worldbank"]
UIS_CFG = PIPE_CFG["uis"]

MIN_INDICATOR_SHARE = PIPE_CFG["coverage"]["min_indicator_share"]

# All indicator nice names in canonical config order
ALL_INDICATORS = list(IND_CFG["indicators"].keys())
MIN_FOR_SCORE  = max(1, int(MIN_INDICATOR_SHARE * len(ALL_INDICATORS)))

OUTDIR = PROJECT_ROOT / "outputs" / "coverage_analytics"
OUTDIR.mkdir(parents=True, exist_ok=True)

print(f"[info] Project root          : {PROJECT_ROOT}")
print(f"[info] Output directory      : {OUTDIR}")
print(f"[info] Year range            : {YEAR_MIN}–{YEAR_MAX}  ({N_YEARS} years)")
print(f"[info] Indicators            : {len(ALL_INDICATORS)}")
print(f"[info] Min for coverage score: {MIN_FOR_SCORE}/{len(ALL_INDICATORS)}")
print(f"[info] UIS access mode       : {UIS_CFG.get('mode','api').upper()}")


# ──────────────────────────────────────────────────────────────
# Step 1 — World Bank country metadata
# ──────────────────────────────────────────────────────────────
def step1_countries() -> tuple[pd.DataFrame, list[str], dict]:
    print("\n[step 1] Fetching World Bank country metadata …")
    countries_df = wb_get_countries(WB_CFG)
    iso2_map     = build_iso2_map(countries_df, WB_CFG.get("iso2_supplement", {}))
    iso_list     = countries_df["iso3c"].tolist()
    print(f"  Non-aggregate economies : {len(countries_df)}")
    print(f"  ISO2 supplement applied : {list(WB_CFG.get('iso2_supplement', {}).keys())}")
    return countries_df, iso_list, iso2_map


# ──────────────────────────────────────────────────────────────
# Step 2 — Pull all indicators from all three sources
# ──────────────────────────────────────────────────────────────
def step2_pull(iso_list: list[str], iso2_map: dict,
               iso_to_name: dict | None = None) -> tuple[pd.DataFrame, pd.DataFrame]:
    """
    Returns
    -------
    raw      : all tidy rows from every source
    src_log  : rows-per-indicator-source summary for provenance
    """
    print(f"\n[step 2] Pulling {len(ALL_INDICATORS)} indicators from WDI / WGI / UIS …")

    wdi_specs = by_source(IND_CFG, "wdi")
    wgi_specs = by_source(IND_CFG, "wgi")
    uis_specs = by_source(IND_CFG, "uis")

    frames:   list[pd.DataFrame] = []
    log_rows: list[dict]          = []

    # ── WDI indicators ──────────────────────────────────────
    print(f"\n  ── WDI ({len(wdi_specs)} indicators) ──")
    for nice, spec in wdi_specs.items():
        print(f"  [wdi] {nice} ← {spec['code']}")
        df = wb_fetch_indicator(
            iso_list, nice, spec["code"], YEAR_MIN, YEAR_MAX,
            iso2_map, WB_CFG, source=None,
        )
        df["_source"] = "wdi"
        print(f"         rows: {len(df)}")
        frames.append(df)
        log_rows.append({"nice": nice, "source": "wdi",
                         "wb_code": spec["code"], "rows": len(df)})

    # ── WGI indicators (source=75) ───────────────────────────
    print(f"\n  ── WGI (source=75, {len(wgi_specs)} indicators) ──")
    for nice, spec in wgi_specs.items():
        print(f"  [wgi] {nice} ← {spec['code']} | source=75")
        df = wb_fetch_indicator(
            iso_list, nice, spec["code"], YEAR_MIN, YEAR_MAX,
            iso2_map, WB_CFG, source="75",
        )
        df["_source"] = "wgi"
        print(f"         rows: {len(df)}")
        frames.append(df)
        log_rows.append({"nice": nice, "source": "wgi",
                         "wb_code": spec["code"], "rows": len(df)})

    # ── UIS indicators ───────────────────────────────────────
    print(f"\n  ── UIS ({len(uis_specs)} indicators, mode={UIS_CFG['mode']}) ──")
    for nice, spec in uis_specs.items():
        print(f"  [uis] {nice} ← {spec['code']}", end="")
        fallback = spec.get("wb_fallback")
        if fallback:
            print(f"  (WDI fallback: {fallback})", end="")
        print()

        bdds_file = spec.get("bdds_file", "both")
        df = uis_fetch(iso_list, nice, spec["code"],
                       YEAR_MIN, YEAR_MAX, UIS_CFG, PROJECT_ROOT,
                       bdds_file=bdds_file, iso_to_name=iso_to_name)
        df["_source"] = "uis"
        print(f"         UIS rows: {len(df)}")
        log_rows.append({"nice": nice, "source": "uis",
                         "uis_code": spec["code"], "rows": len(df)})

        # Fill gaps from WDI fallback
        if fallback and len(df) > 0:
            # Pull WDI fallback for all countries
            df_wb = wb_fetch_indicator(
                iso_list, nice, fallback, YEAR_MIN, YEAR_MAX,
                iso2_map, WB_CFG, source=None,
            )
            df_wb["_source"] = "wdi_fallback"
            log_rows.append({"nice": nice, "source": "wdi_fallback",
                             "wb_code": fallback, "rows": len(df_wb)})

            # Only use WDI rows where UIS has no data for that country-year
            uis_keys = set(zip(df["iso3c"], df["year"]))
            df_fill  = df_wb[~df_wb.apply(
                lambda r: (r["iso3c"], r["year"]) in uis_keys, axis=1
            )].copy()
            print(f"         WDI fallback fill rows: {len(df_fill)}")
            df = pd.concat([df, df_fill], ignore_index=True)
        elif fallback and len(df) == 0:
            # UIS returned nothing — use WDI fully
            print(f"  [warn] UIS returned 0 rows for {nice}; using WDI fallback only.")
            df_wb = wb_fetch_indicator(
                iso_list, nice, fallback, YEAR_MIN, YEAR_MAX,
                iso2_map, WB_CFG, source=None,
            )
            df_wb["_source"] = "wdi_fallback"
            log_rows.append({"nice": nice, "source": "wdi_fallback",
                             "wb_code": fallback, "rows": len(df_wb)})
            df = df_wb

        frames.append(df)

    if not frames:
        raise SystemExit("[fatal] No data returned from any source.")

    raw = pd.concat(frames, ignore_index=True)

    # ── Mark availability and deduplicate ───────────────────
    raw["avail"] = raw["value"].notna().astype(int)
    raw = deduplicate(raw)
    print(f"\n  Total rows (after dedup): {len(raw):,}")

    src_log = pd.DataFrame(log_rows)
    return raw, src_log


# ──────────────────────────────────────────────────────────────
# Step 3 — Build coverage matrices
# ──────────────────────────────────────────────────────────────
def step3_coverage(
    raw: pd.DataFrame,
    countries_df: pd.DataFrame,
) -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    print("\n[step 3] Building coverage matrices …")

    # Aggregate to max-per-country-year-indicator
    observed = (raw.groupby(["iso3c", "country", "nice", "year"], as_index=False)
                .agg(avail=("avail", "max")))

    # Full Cartesian grid: every country × indicator × year
    cg = countries_df[["iso3c", "country"]].drop_duplicates()
    ig = pd.DataFrame({"nice": ALL_INDICATORS})
    yg = pd.DataFrame({"year": YEARS})

    full_grid = (
        cg.assign(_k=1)
        .merge(ig.assign(_k=1), on="_k")
        .merge(yg.assign(_k=1), on="_k")
        .drop(columns="_k")
    )

    # Long availability (0/1 per country-indicator-year)
    # Merge on iso3c+nice+year only — country column in UIS BDDS rows
    # contains ISO3 codes not display names, so including it in the key
    # causes silent row drops. Country name comes from full_grid (countries_df).
    observed_key = observed.drop(columns=["country"], errors="ignore")
    coverage_long = (
        full_grid
        .merge(observed_key, on=["iso3c", "nice", "year"], how="left")
        .assign(avail=lambda d: d["avail"].fillna(0).astype(int))
        .merge(countries_df[["iso3c", "income_group"]], on="iso3c", how="left")
    )
    coverage_long = sort_by_income(coverage_long, ["country", "nice", "year"])

    # Per-country-indicator summary
    cov_ind = (
        coverage_long
        .groupby(["iso3c", "country", "income_group", "nice"], as_index=False)
        .agg(years_available=("avail", "sum"))
    )
    cov_ind["coverage_share"] = cov_ind["years_available"] / N_YEARS
    cov_ind = sort_by_income(cov_ind, ["country", "nice"])

    # Wide matrix
    cov_wide = (
        cov_ind.pivot(
            index=["iso3c", "country", "income_group"],
            columns="nice",
            values="coverage_share",
        )
        .reset_index()
    )
    for name in ALL_INDICATORS:
        if name not in cov_wide.columns:
            cov_wide[name] = 0.0

    cov_wide["indicators_observed"] = (
        cov_wide[ALL_INDICATORS].gt(0).sum(axis=1)
    )

    # overall_coverage: mean of per-indicator shares, suppressed when too sparse
    mean_cov = cov_wide[ALL_INDICATORS].mean(axis=1, skipna=True)
    sufficient = cov_wide["indicators_observed"] >= MIN_FOR_SCORE
    cov_wide["overall_coverage"] = mean_cov.where(sufficient, other=np.nan)

    # Sort: income tier → overall_coverage descending → country
    cov_wide = (
        cov_wide
        .assign(_isort=lambda d: d["income_group"].map(INCOME_ORDER).fillna(4))
        .sort_values(["_isort", "overall_coverage", "country"],
                     ascending=[True, False, True])
        .drop(columns="_isort")
        .reset_index(drop=True)
    )

    return coverage_long, cov_ind, cov_wide


# ──────────────────────────────────────────────────────────────
# Step 4 — Rank countries per income tier
# ──────────────────────────────────────────────────────────────
def rank_countries(cov_wide: pd.DataFrame, k: int) -> pd.DataFrame:
    tiers = PIPE_CFG["selection"]["tiers"]
    keep  = (["rank_in_group", "iso3c", "country", "income_group",
               "indicators_observed", "overall_coverage"] + ALL_INDICATORS)

    frames = []
    for tier in tiers:
        sub = (cov_wide[cov_wide["income_group"] == tier]
               .dropna(subset=["overall_coverage"])
               .sort_values(["overall_coverage", "country"], ascending=[False, True])
               .head(k).copy())
        sub.insert(0, "rank_in_group", range(1, len(sub) + 1))
        frames.append(sub[[c for c in keep if c in sub.columns]])

    return pd.concat(frames, ignore_index=True) if frames else pd.DataFrame()


# ──────────────────────────────────────────────────────────────
# Step 5 — Save all outputs
# ──────────────────────────────────────────────────────────────
def step5_save(
    raw:           pd.DataFrame,
    src_log:       pd.DataFrame,
    coverage_long: pd.DataFrame,
    cov_ind:       pd.DataFrame,
    cov_wide:      pd.DataFrame,
) -> None:
    print("\n[step 4] Saving outputs …")

    def save(df: pd.DataFrame, name: str) -> None:
        p = OUTDIR / name
        df.to_csv(p, index=False)
        print(f"  [OK] {name}  ({len(df):,} rows)")

    # Raw long with source tags
    save(raw[["iso3c", "country", "nice", "year", "value", "_source", "avail"]],
         f"raw_long_{PERIOD_TAG}.csv")

    # Source pull log
    save(src_log, f"data_source_log_{PERIOD_TAG}.csv")

    # Long availability
    save(coverage_long, f"coverage_long_{PERIOD_TAG}.csv")

    # Per-country-indicator summary
    save(cov_ind, f"coverage_by_indicator_{PERIOD_TAG}.csv")

    # Wide matrix (proportions)
    save(cov_wide, f"coverage_matrix_{PERIOD_TAG}.csv")

    # Wide matrix (percentages for paper appendix)
    pct = cov_wide[["iso3c", "country", "income_group",
                    "indicators_observed", "overall_coverage"] + ALL_INDICATORS].copy()
    for col in ["overall_coverage"] + ALL_INDICATORS:
        if col in pct.columns:
            pct[col] = (pct[col] * 100).round(1)
    save(pct, f"coverage_matrix_pct_{PERIOD_TAG}.csv")

    # Income-tier rankings
    for k, label in [(3, "top3"), (10, "top10")]:
        ranked = rank_countries(cov_wide, k)
        if not ranked.empty:
            save(ranked, f"{label}_by_income_group_{PERIOD_TAG}.csv")
            _print_ranking(ranked, k)

    # Excel workbook
    _save_excel(cov_wide)

    print(f"\n[done] All outputs: {OUTDIR.resolve()}")
    print("\n[next steps]")
    print("  1. Review top3_by_income_group CSV")
    print("  2. Decide on final 12 countries (or accept top-3 per tier)")
    print("  3. Run: python src/02_clean_merge_panel.py")


def _print_ranking(ranked: pd.DataFrame, k: int) -> None:
    for tier in ranked["income_group"].unique():
        sub = ranked[ranked["income_group"] == tier]
        print(f"\n  {'─'*55}")
        print(f"  Top {k}: {tier}")
        print(f"  {'─'*55}")
        for _, row in sub.iterrows():
            cov = f"{row['overall_coverage']:.1%}" if pd.notna(row["overall_coverage"]) else "n/a"
            obs = int(row["indicators_observed"])
            print(f"    {int(row['rank_in_group']):2d}. {row['iso3c']}  "
                  f"{row['country']:<32s}  coverage={cov}  "
                  f"indicators={obs}/{len(ALL_INDICATORS)}")


def _save_excel(cov_wide: pd.DataFrame) -> None:
    try:
        import openpyxl  # noqa: F401
    except ImportError:
        print("  [skip] Excel export — openpyxl not installed.")
        return
    p = OUTDIR / f"coverage_matrix_{PERIOD_TAG}.xlsx"
    with pd.ExcelWriter(p, engine="openpyxl") as writer:
        cov_wide.to_excel(writer, sheet_name="All", index=False)
        for tier in PIPE_CFG["selection"]["tiers"]:
            sname = tier[:31].replace("/", "-").replace(":", "-")
            cov_wide[cov_wide["income_group"] == tier].to_excel(
                writer, sheet_name=sname, index=False)
    print(f"  [OK] coverage_matrix_{PERIOD_TAG}.xlsx  (multi-sheet workbook)")


# ──────────────────────────────────────────────────────────────
# Main
# ──────────────────────────────────────────────────────────────
def main() -> None:
    countries_df, iso_list, iso2_map = step1_countries()
    # Build iso3c → display name lookup for BDDS country name fix
    iso_to_name = dict(zip(countries_df["iso3c"], countries_df["country"]))
    raw, src_log                     = step2_pull(iso_list, iso2_map,
                                                  iso_to_name=iso_to_name)
    coverage_long, cov_ind, cov_wide = step3_coverage(raw, countries_df)
    step5_save(raw, src_log, coverage_long, cov_ind, cov_wide)


if __name__ == "__main__":
    main()